In [1]:
import pandas as pd
import torch
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_from_disk
import os

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load base model
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct",device_map={"": 1}) # Use GPU 1)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
print("✅ Model loaded successfully!")

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.38it/s]


✅ Model loaded successfully!


In [3]:
def test_model(patient_id, query, max_length=2000):
    """Test model with mental health query"""
    # Format exactly like training data
    messages = [
        {
            "role": "system",
            "content": """You are a helpful mental health counselling assistant, please answer the mental health questions based on the patient's description. \nThe assistant gives helpful, comprehensive, and appropriate answers to the user's questions."""
        },
        {
            "role": "user",
            "content": f"Query: {query}"
        }
    ]
    
    # print(model.device)

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        
    ).to(model.device)
    
    # Tokenize and generate
    outputs = model.generate(
        **inputs, 
        max_new_tokens=max_length,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id)

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])
    
    # Clean up the eos for clear response
    response = response.replace(tokenizer.eos_token, "").strip()
    
    return response

print("✅ test_model function defined!")

✅ test_model function defined!


In [4]:
# prediction = test_model("UID_TEST", "I am feeling very tired")
# print(prediction)

In [5]:
def parse_dataset_sample(sample):
    """Parse your dataset sample to extract query and ground truth response"""
    return {
        'query': sample['query'],
        'ground_truth': sample['response'],
        'instruction': sample['instruction']
    }

print("✅ Dataset parser function defined!")

✅ Dataset parser function defined!


In [ ]:
# Load test dataset
print("📊 Loading mental health test dataset...")

# Load your test dataset
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
dataset = load_from_disk(f"{dataset_based_path}/mh-mental-chat-16k_v1")
test_dataset = dataset["test"]

# Take a subset for evaluation (adjust size as needed)
# test_subset = test_dataset.select(range(2))  # Adjust size as needed
test_subset = test_dataset

print(f"🧪 Generating predictions for {len(test_subset)} test samples...")

# Prepare data storage
evaluation_data = []

for i, sample in enumerate(test_subset):
    # Parse the sample using your dataset structure
    parsed = parse_dataset_sample(sample)
    query = parsed['query']
    ground_truth = parsed['ground_truth']
    
    # Generate prediction using your model
    prediction = test_model("UID_TEST", query)
    
    # Store all data
    evaluation_data.append({
        'sample_id': i,
        'query': query,
        'ground_truth': ground_truth,
        'prediction': prediction,
        'instruction': parsed['instruction']
    })
    
    if (i + 1) % 10 == 0:
        print(f"✅ Processed {i + 1}/{len(test_subset)} samples...")

print(f"🎯 Generated {len(evaluation_data)} predictions")

📊 Loading mental health test dataset...
🧪 Generating predictions for 1606 test samples...


✅ Processed 10/1606 samples...
✅ Processed 20/1606 samples...
✅ Processed 30/1606 samples...
✅ Processed 40/1606 samples...
✅ Processed 50/1606 samples...
✅ Processed 60/1606 samples...
✅ Processed 70/1606 samples...
✅ Processed 80/1606 samples...
✅ Processed 90/1606 samples...
✅ Processed 100/1606 samples...
✅ Processed 110/1606 samples...
✅ Processed 120/1606 samples...
✅ Processed 130/1606 samples...
✅ Processed 140/1606 samples...
✅ Processed 150/1606 samples...
✅ Processed 160/1606 samples...
✅ Processed 170/1606 samples...
✅ Processed 180/1606 samples...
✅ Processed 190/1606 samples...
✅ Processed 200/1606 samples...
✅ Processed 210/1606 samples...
✅ Processed 220/1606 samples...
✅ Processed 230/1606 samples...
✅ Processed 240/1606 samples...
✅ Processed 250/1606 samples...
✅ Processed 260/1606 samples...
✅ Processed 270/1606 samples...
✅ Processed 280/1606 samples...
✅ Processed 290/1606 samples...
✅ Processed 300/1606 samples...
✅ Processed 310/1606 samples...
✅ Processed 320/1

In [ ]:
# Create DataFrame and save to CSV
df = pd.DataFrame(evaluation_data)

# Create output directory if it doesn't exist
output_dir = "/home/jovyan/Works/MSc_Thesis/mh_conversation/model_evaluation/results/base_v1"
os.makedirs(output_dir, exist_ok=True)

# Save to CSV
csv_filename = f"{output_dir}/mh_predictions_and_ground_truth.csv"
df.to_csv(csv_filename, index=False, encoding='utf-8')

print(f"✅ Saved evaluation data to: {csv_filename}")
print(f"📊 Dataset shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")

# Display sample
print("\n🔍 Sample data:")
print(df.head(2))

✅ Saved evaluation data to: /home/jovyan/Works/MSc_Thesis/mh_conversation/model_evaluation/results/base_v1/mh_predictions_and_ground_truth.csv
📊 Dataset shape: (2, 5)
📋 Columns: ['sample_id', 'query', 'ground_truth', 'prediction', 'instruction']

🔍 Sample data:
   sample_id                                              query  \
0          0  Throughout our counseling session, I would lik...   
1          1  (sighing deeply) I've been feeling overwhelmed...   

                                        ground_truth  \
0  Managing stress-related challenges can be a di...   
1  (leaning forward, offering a sympathetic smile...   

                                          prediction  \
0  I'm so glad you've reached out for support. It...   
1  I can sense the frustration and guilt you're f...   

                                         instruction  
0  You are a helpful mental health counselling as...  
1  You are a helpful mental health counselling as...  


In [ ]:
# Verify the saved file
print("🔍 Verifying saved CSV file...")

# Load and check the saved data
loaded_df = pd.read_csv(csv_filename)
print(f"✅ Successfully loaded CSV with shape: {loaded_df.shape}")

# Show statistics
print(f"\n📊 Data Statistics:")
print(f"   Queries: {loaded_df['query'].notna().sum()}")
print(f"   Ground truths: {loaded_df['ground_truth'].notna().sum()}")
print(f"   Predictions: {loaded_df['prediction'].notna().sum()}")

print(f"\n💾 CSV file saved at: {csv_filename}")
print("🎉 Prediction generation complete!")